# CAPR-Boost: training in Google Colab

Is notebook me teen kaam hote hain:

| Part | Kya hota hai | Time (Colab free CPU) |
|---|---|---|
| **A. Live demo** | ek model train karke accuracy / F1, comparison, confusion matrix, SHAP | ~2–4 min |
| **B. Paper numbers reproduce** | paper wale same seeds aur splits par dobara training, paper se diff table | headline ~5–8 min |
| **C. Deployment model export** | web app ke liye final models train karke `capr_models.zip` download | ~3–5 min |

**Steps:** `Runtime → Run all` dabao. Cell 2 me `capr_colab_package.zip` upload karna hoga
(ya Google Drive se lena ho to `USE_DRIVE = True` karo). GPU ki zarurat nahi.

## 1. Libraries

In [ ]:
# same library versions as the paper experiments and the Docker ML service
!pip -q install "lightgbm==4.7.0" "scikit-learn==1.9.1" pyyaml
import os, time, sys, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import lightgbm, sklearn
print("lightgbm", lightgbm.__version__, "| scikit-learn", sklearn.__version__, "| CPUs", os.cpu_count())

## 2. Project files load karo
`capr_colab_package.zip` (submission folder me `08_Colab_Demo/` ke andar) upload karo.
Drive use karna ho to zip ko Drive me rakho, `USE_DRIVE = True` karo aur `DRIVE_ZIP` ka path set karo.

In [ ]:
USE_DRIVE = False
DRIVE_ZIP = "/content/drive/MyDrive/capr_colab_package.zip"

if not os.path.exists("/content/capr-anomaly/src"):
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        zpath = DRIVE_ZIP
    else:
        from google.colab import files
        up = files.upload()
        zpath = list(up.keys())[0]
    zipfile.ZipFile(zpath).extractall("/content")
os.chdir("/content/capr-anomaly")
sys.path.insert(0, "/content/capr-anomaly")
os.environ["CAPR_NJOBS"] = str(os.cpu_count())
print(os.listdir("data"))

# Part A: Live demo
## 3. Dataset load + split (test set 30%, fixed seed 42)

In [ ]:
import yaml
from src.data import load_dataset, make_split
from src.metrics import evaluate

DATASET = "darknet2020"          # ya "ids2018"
LABEL_RATIO = 0.05               # sirf 5% training flows ke labels use honge
SEED = 0

cfg_file = "configs/v2_darknet.yaml" if DATASET == "darknet2020" else "configs/v2_ids2018.yaml"
cfg = yaml.safe_load(open(cfg_file))
dcfg = [d for d in cfg["datasets"] if d["name"] == DATASET][0]
t = time.time()
X, y, feats = load_dataset(dcfg)
print(f"flows={len(y):,}  features={X.shape[1]}  anomaly share={y.mean():.3f}  ({time.time()-t:.1f}s)")

sp = make_split(X, y, LABEL_RATIO, 0.0, SEED)
print(f"labelled={len(sp['y_l']):,}  unlabelled={len(sp['X_u']):,}  test={len(sp['y_te']):,}")

## 4. CAPR-Boost live training

In [ ]:
from src.capr_boost import CAPRBoost

t = time.time()
model = CAPRBoost(seed=SEED).fit(sp["X_l"], sp["y_l"], sp["X_u"])
train_time = time.time() - t
print(f"training time: {train_time:.1f} s   chosen variant: {model.variant}")
print("variant selection CV accuracy:", {k: round(v, 4) for k, v in model.history[0].items() if k.startswith("cv_")})

p = model.score(sp["X_te"])
res = evaluate(sp["y_te"], p, (p >= 0.5).astype(int))
pd.Series({k: res[k] for k in ["accuracy", "precision", "recall", "f1", "auc_roc"]}).mul(100).round(2).to_frame("CAPR-Boost (%)")

## 5. Baselines se comparison (same split)

In [ ]:
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier

rows = []
def add(name, prob, secs):
    r = evaluate(sp["y_te"], prob, (prob >= 0.5).astype(int))
    rows.append({"method": name, "accuracy": r["accuracy"]*100, "precision": r["precision"]*100,
                 "recall": r["recall"]*100, "f1": r["f1"]*100, "train_s": secs})

add("CAPR-Boost (proposed)", p, train_time)
t = time.time()
m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.04, num_leaves=63, min_child_samples=5,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.7, random_state=SEED, verbose=-1)
m.fit(sp["X_l"], sp["y_l"]); add("LightGBM", m.predict_proba(sp["X_te"])[:, 1], time.time()-t)
t = time.time()
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED).fit(sp["X_l"], sp["y_l"])
add("Random Forest", rf.predict_proba(sp["X_te"])[:, 1], time.time()-t)
table = pd.DataFrame(rows).round(2)
table

### Confusion matrix + top features

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ConfusionMatrixDisplay.from_predictions(sp["y_te"], (p >= 0.5).astype(int), display_labels=["Normal", "Anomaly"],
                                        cmap="Blues", values_format=",", ax=ax[0])
ax[0].set_title("CAPR-Boost, test set")
imp = pd.Series(model.feature_importance(), index=feats).sort_values().tail(12)
imp.plot.barh(ax=ax[1], color="#2a78d6"); ax[1].set_title("Top features (gain share)")
plt.tight_layout(); plt.show()

### Ek flow ka explanation (TreeSHAP) + uncertainty

In [ ]:
mean, std = model.predict_with_uncertainty(sp["X_te"])
i = int(np.argmax(mean))
contrib, base = model.contributions(sp["X_te"][i:i+1])
top = pd.Series(contrib[0], index=feats).sort_values(key=abs, ascending=False).head(8)
print(f"flow #{i}: anomaly probability={mean[i]:.3f}, uncertainty(std)={std[i]:.3f}, true label={sp['y_te'][i]}")
top.sort_values().plot.barh(color=["#eb6834" if v > 0 else "#2a78d6" for v in top.sort_values()], figsize=(7, 3.5))
plt.title("SHAP contribution (positive = towards anomaly)"); plt.tight_layout(); plt.show()

## 6. (Optional, ~2–3 min) Label noise demo
Training labels me 20% galat labels daal kar dekhte hain. Test labels kabhi change nahi hote.

In [ ]:
spn = make_split(X, y, LABEL_RATIO, 0.20, SEED)
t = time.time(); mb = CAPRBoost(seed=SEED).fit(spn["X_l"], spn["y_l"], spn["X_u"]); tb = time.time()-t
pb = mb.score(spn["X_te"])
ml = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.04, num_leaves=63, min_child_samples=5, subsample=0.8,
                        subsample_freq=1, colsample_bytree=0.7, random_state=SEED, verbose=-1).fit(spn["X_l"], spn["y_l"])
pl = ml.predict_proba(spn["X_te"])[:, 1]
out = []
for name, pr in [("CAPR-Boost", pb), ("LightGBM (no noise handling)", pl)]:
    r = evaluate(spn["y_te"], pr, (pr >= 0.5).astype(int))
    out.append({"method": name, "accuracy": r["accuracy"]*100, "f1": r["f1"]*100})
print("variant chosen:", mb.variant, "| labels flipped back:", mb.history[0]["flipped"], f"| {tb:.1f}s")
pd.DataFrame(out).round(2)

# Part B: Paper ke numbers reproduce karo
`reproduce.py` wahi config, wahi seeds aur wahi train/test split use karta hai jo paper me use hue the,
aur result ko `results/report/raw_results_all.csv` (paper ke raw numbers) se compare karta hai.
`diff = 0.00` ka matlab paper ka number exactly reproduce hua.

| PRESET | Kya chalta hai | Paper value | Time |
|---|---|---|---|
| `headline` | Darknet2020, 5% labels, 5 seeds: CAPR-Boost, LightGBM, XGBoost | CAPR-Boost Acc 96.94 / F1 90.99 | ~5–8 min |
| `ids_headline` | IDS2018, 0.5% & 5% labels, 3 seeds | Acc 99.64 / 99.87 | ~10 min |
| `noise` | Darknet2020, 5% labels, 20% label noise, 3 seeds | CAPR-Boost F1 78.70 vs LightGBM 67.14 | ~5 min |
| `all_capr_boost` | CAPR-Boost ke saare paper numbers (dono datasets, saare ratios, noise) | Tables 7.2–7.8 | ~1–2 hours |
| `full` | paper ka poora experiment (sab baselines) | sab tables | bahut ghante |

In [ ]:
PRESET = "headline"
!python reproduce.py --preset {PRESET} --out results_repro
pd.read_csv("results_repro/comparison_with_paper.csv")

# Part C: Web app ke liye final model train + export
Yeh wahi model hai jo web app use karta hai (5% labels, seed 0). Expected test result:
Darknet2020 accuracy 97.15%, F1 91.70%; IDS2018 accuracy 99.90%, F1 99.75%.
Download hone wali `capr_models.zip` ko `capr-webapp/ml-service/artifacts/` me extract karo (HOW_TO_RUN.txt section C). Phir `docker compose restart ml-service` aur web app me Models -> "Sync from ml-service".

In [ ]:
import json, shutil
!python train_final.py --config configs/v2_darknet.yaml --dataset darknet2020 --label_ratio 0.05 --seed 0 --out export_artifacts
!python train_final.py --config configs/v2_ids2018.yaml --dataset ids2018 --label_ratio 0.05 --seed 0 --out export_artifacts
for d in ["darknet2020", "ids2018"]:
    m = json.load(open(f"export_artifacts/{d}/meta.json"))
    print(d, "variant:", m["variant"], {k: round(v * 100, 2) for k, v in m["test_metrics"].items()
                                         if k in ("accuracy", "precision", "recall", "f1")})
shutil.make_archive("capr_models", "zip", "export_artifacts")
print("capr_models.zip ready")

In [ ]:
try:
    from google.colab import files
    files.download("capr_models.zip")
except ImportError:
    print("not in Colab: file saved as capr_models.zip")

---
**Note:** Part A ek seed (seed 0) ka result hai, isliye yeh paper ke multi-seed average se thoda alag hota hai
(Darknet2020: seed 0 = 97.15% accuracy, 5 seeds ka average = 96.94%). Paper ka exact number Part B deta hai.
Agar Colab ki numpy/pandas version alag ho to kabhi-kabhi last decimal me ±0.01–0.05 ka fark aa sakta hai.